# Delhi Housing Price Predictor: Data Cleaning and Model Training

Trains a regression model on Delhi housing listings (`raw_magicbricks.csv`) to predict price from locality, BHK, area, furnishing, and related features.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [ ]:
from google.colab import files
uploaded = files.upload()
filename = list(uploaded.keys())[0]

In [ ]:
df = pd.read_csv(filename)
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df['Locality'].nunique(), df['Locality'].astype(str).str.len().describe()

## Cleaning

In [ ]:
df_clean = df.copy()

# rows with no Furnishing/Type are unusable, drop them
df_clean = df_clean[df_clean['Furnishing'].notna() & df_clean['Type'].notna()]

In [ ]:
# a scraping bug dumped the full listing description into Locality for some rows
# instead of an actual area name; these are identifiable by length and are dropped
df_clean = df_clean[df_clean['Locality'].astype(str).str.len() <= 60]

In [ ]:
def extract_area(locality: str) -> str:
    return locality.split(',')[-1].strip()

df_clean['Locality'] = df_clean['Locality'].astype(str).apply(extract_area)

In [ ]:
# with 365 distinct localities across 1259 rows, one hot encoding directly
# would leave most categories with a handful of samples; bucket rare ones
MIN_LISTINGS = 5
locality_counts = df_clean['Locality'].value_counts()
common_localities = locality_counts[locality_counts >= MIN_LISTINGS].index
df_clean['Locality'] = df_clean['Locality'].where(df_clean['Locality'].isin(common_localities), 'Other')
df_clean['Locality'].nunique()

In [ ]:
# Per_Sqft is derived from Price / Area, keeping it as a feature would leak the target
df_clean = df_clean.drop(columns=['Per_Sqft'])

In [ ]:
for col in ['Bathroom', 'Parking']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

In [ ]:
price_per_sqft = df_clean['Price'] / df_clean['Area']
low, high = price_per_sqft.quantile([0.01, 0.99])
df_clean = df_clean[(price_per_sqft >= low) & (price_per_sqft <= high)]
df_clean.shape

In [ ]:
df_clean.to_csv('clean_listings.csv', index=False)

## Train/test split and preprocessing

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

target_col = 'Price'
numeric_features = ['Area', 'BHK', 'Bathroom', 'Parking']
categorical_features = ['Locality', 'Furnishing', 'Type', 'Status', 'Transaction']

X = df_clean[numeric_features + categorical_features]
y = df_clean[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
])

## Baseline: Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

baseline_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', LinearRegression()),
])

baseline_pipeline.fit(X_train, y_train)
baseline_preds = baseline_pipeline.predict(X_test)

print('MAE:', mean_absolute_error(y_test, baseline_preds))
print('RMSE:', mean_squared_error(y_test, baseline_preds, squared=False))
print('R2:', r2_score(y_test, baseline_preds))

## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestRegressor(n_estimators=300, random_state=42)),
])

rf_pipeline.fit(X_train, y_train)
rf_preds = rf_pipeline.predict(X_test)

print('MAE:', mean_absolute_error(y_test, rf_preds))
print('RMSE:', mean_squared_error(y_test, rf_preds, squared=False))
print('R2:', r2_score(y_test, rf_preds))

## Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', GradientBoostingRegressor(random_state=42)),
])

gb_pipeline.fit(X_train, y_train)
gb_preds = gb_pipeline.predict(X_test)

print('MAE:', mean_absolute_error(y_test, gb_preds))
print('RMSE:', mean_squared_error(y_test, gb_preds, squared=False))
print('R2:', r2_score(y_test, gb_preds))

## Final model

In [ ]:
final_pipeline = rf_pipeline

sample = pd.DataFrame([{
    'Area': 1200, 'BHK': 3, 'Bathroom': 2, 'Parking': 1,
    'Locality': 'Dwarka', 'Furnishing': 'Semi-Furnished',
    'Type': 'Apartment', 'Status': 'Ready_to_move', 'Transaction': 'Resale',
}])
final_pipeline.predict(sample)[0]

In [ ]:
import joblib
joblib.dump(final_pipeline, 'price_model.pkl')

In [ ]:
from google.colab import files
files.download('price_model.pkl')
files.download('clean_listings.csv')